# Assignment 5: Air sensor bias correction model validation

Low-cost air quality sensors are cheap enough to deploy in dense networks, which is exactly
what you want for measuring pollution exposure at neighbourhood scale. The problem is that
they are not very good. They drift, they respond to temperature and humidity as much as to
the pollutant, and their raw output is not in physical units at all.

The standard fix is **calibration**: co-locate the cheap sensor with a reference-grade
analyzer, learn the mapping between them, and apply that correction to the sensor's readings
elsewhere. That mapping is a machine learning model — and validating it properly is much
harder than it looks, which is the real subject of this assignment.

You will use the dataset of De Vito et al. (2008): a multisensor device containing five
metal-oxide gas sensors, deployed for just over a year at road level in an Italian city,
alongside certified reference analyzers.

Answer each numbered question in the empty cell below it.

In [ ]:
import io
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

## Load the data

The archive is hosted by the UCI Machine Learning Repository. Missing values are coded
`-200`, which we convert to `NaN` on load — if you leave them in, every model you fit will
be quietly ruined.

In [ ]:
url = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"
with urllib.request.urlopen(url) as resp:
    archive = zipfile.ZipFile(io.BytesIO(resp.read()))

# European CSV conventions: semicolon separator, comma decimal mark.
aq = pd.read_csv(archive.open("AirQualityUCI.csv"), sep=";", decimal=",",
                 usecols=range(15), na_values=[-200])
aq = aq.dropna(how="all")

aq["datetime"] = pd.to_datetime(
    aq["Date"] + " " + aq["Time"].str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S")
aq = aq.sort_values("datetime").reset_index(drop=True)

print(f"{len(aq)} hourly records, {aq.datetime.min()} to {aq.datetime.max()}")
aq.head()

The columns divide into three groups:

- **`PT08.S1` through `PT08.S5`** — the raw responses of the five low-cost metal-oxide
  sensors. These are the cheap instrument.
- **`CO(GT)`, `NMHC(GT)`, `C6H6(GT)`, `NOx(GT)`, `NO2(GT)`** — the co-located reference
  analyzer values. `GT` stands for ground truth. This is what you are trying to predict.
- **`T`, `RH`, `AH`** — temperature, relative humidity, absolute humidity.

We will focus on **NOx**: sensor `PT08.S3(NOx)` against reference `NOx(GT)`.

## Part 1: How bad is the raw sensor?

1) Plot the raw sensor response `PT08.S3(NOx)` against the reference `NOx(GT)`. Compute the
correlation. You should find it is **negative** — explain why, given that these sensors work
by measuring a change in electrical resistance.

2) Plot both series against time for a two-week window. Do they track each other?

3) How many hours have a valid reading for both sensor and reference? What fraction of the deployment is that, and what does the pattern of missingness look like over time?

## Part 2: Baselines

Before fitting anything, establish what has to be beaten.

4) **Baseline 1 — raw sensor.** The raw sensor is in the wrong units entirely, so scoring it
directly is meaningless. Instead, fit a single-variable linear regression from
`PT08.S3(NOx)` to `NOx(GT)` on the full dataset, and report RMSE and $R^2$. This is the
simplest possible calibration, and it is what a manufacturer's default correction looks like.

5) **Baseline 2 — predict the mean.** Report the RMSE of always predicting the mean reference NOx. Any model scoring worse than this is worse than useless.

## Part 3: A better correction model

Metal-oxide sensors are cross-sensitive: they respond to other gases, and strongly to
temperature and humidity. A correction using more inputs should do better.

6) Build a feature matrix from all five `PT08.S*` sensor channels plus `T`, `RH` and `AH`,
with `NOx(GT)` as the target. Drop rows with missing values and report how many remain.

7) Fit a `RandomForestRegressor` using a **random** train/test split (`random_state=0`). Report RMSE and $R^2$ on the test set.

8) Compare against both baselines. How much has the correction improved things?

## Part 4: The validation problem

That test score is almost certainly too good, and this is the heart of the assignment.

These are **hourly measurements**. Air pollution is highly autocorrelated — the NOx
concentration at 3pm is very close to the value at 2pm and at 4pm. A random split scatters
those neighbouring hours across training and test, so for almost every test hour the model
has already seen the hours either side of it.

That is not the task. In deployment, you calibrate a sensor over some period and then apply
the correction *going forward*, to hours the model has never seen.

9) Re-split the data **chronologically**: train on the first 70% of the record in time, test
on the last 30%. Fit the same random forest and report RMSE and $R^2$.

10) Report both scores side by side. How large is the gap between the random split and the chronological split?

11) Now do the same comparison for the simple linear calibration from question 4. Is the gap as large? Explain why a flexible model is more vulnerable to this than a rigid one.

12) Plot predicted against observed for the chronological test period, as a time series. Does the error grow as you move further from the training period? What would that imply for how often a deployed sensor needs recalibrating?

## Part 5: Leakage audit

13) Suppose you had standardized the features using `StandardScaler` fitted on the **whole**
dataset before splitting. Explain precisely what information would leak, and why fitting the
scaler inside a `Pipeline` prevents it.

*Write your answer here.*

14) Demonstrate it. Fit a scaler on the full dataset, then split and fit a model; separately, do the same using a `Pipeline` so the scaler is fitted only on training folds. Report both scores.

15) The dataset contains a `NMHC(GT)` column that is missing for most of the record. If you left it in your feature matrix and dropped rows with missing values, what would happen to your sample — and would your reported score still describe the deployment you care about?

*Write your answer here.*

## Part 6: Report honestly

16) Write a short summary — five or six sentences — stating what performance a user of this
corrected sensor should actually expect. Give the number you would stand behind, say which
validation scheme produced it, and state one thing you could not determine from this dataset.

*Write your answer here.*

```{admonition} A note on what counts as success
:class: tip
If your honest score is substantially worse than your first random-split score, that is the
correct result and you should report it as such. The purpose of this assignment is to
measure that gap, not to make it disappear.
```